#En esta actividad estaré trabjando con un dataset de hongos que tiene el propósito de predecir el tipo de hongo que es (venenoso, no venenoso y no recomendado).

Mushroom [Dataset]. (1981). UCI Machine Learning Repository. https://doi.org/10.24432/C5959T.

##En este documento se hace la limpieza del dataset eliminando columnas que no aportan información al igual que columnas que tengan menos del 70% de sus filas completas.

##Una vez limpio el data set se hace la transformación de las columnas no numéricas mediante dummies (one hot encoding).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Import, según la documentación de UC Irvine

In [ ]:
pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

mushroom = fetch_ucirepo(id=73)

X = mushroom.data.features
y = mushroom.data.targets

print(mushroom.metadata)

print(mushroom.variables)


{'uci_id': 73, 'name': 'Mushroom', 'repository_url': 'https://archive.ics.uci.edu/dataset/73/mushroom', 'data_url': 'https://archive.ics.uci.edu/static/public/73/data.csv', 'abstract': 'From Audobon Society Field Guide; mushrooms described in terms of physical characteristics; classification: poisonous or edible', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 8124, 'num_features': 22, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['poisonous'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1981, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5959T', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "This data set includes descriptions of hypothetical samples corresponding to 23 species of gilled mushrooms in the Agaricus and Lepiota Family (pp. 500-525).  Each species is identified as definitely edible, definitely po

#Nota: Menciona que hay missing values en el feature 11 "stalk root" y nos dice que cap-color y veil-type son binarias

Modificar la importación para manejarlo como pandas: ¿por qué? poruqe estoy más familiarizado y me gusta para analizar los datos

In [ ]:
#pasar a pandas
X = pd.DataFrame(mushroom.data.features)
y = pd.DataFrame(mushroom.data.targets)

# estoy juntando el target con el resto de datos en una tabla (más adelante se vuelven a separar)
df = pd.concat([X, y], axis=1)

print(df.head())

  cap-shape cap-surface cap-color bruises odor gill-attachment gill-spacing  \
0         x           s         n       t    p               f            c   
1         x           s         y       t    a               f            c   
2         b           s         w       t    l               f            c   
3         x           y         w       t    p               f            c   
4         x           s         g       f    n               f            w   

  gill-size gill-color stalk-shape  ... stalk-color-above-ring  \
0         n          k           e  ...                      w   
1         b          k           e  ...                      w   
2         b          n           e  ...                      w   
3         n          n           e  ...                      w   
4         b          k           t  ...                      w   

  stalk-color-below-ring veil-type veil-color ring-number ring-type  \
0                      w         p          w           o

Me da 23 columnas, sin embargo solo hay 22 features, agregó una columna de id, la cual no es del data frame.

In [ ]:
df.shape

(8124, 23)

In [ ]:
df.describe()

,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,poisonous
count,8124,8124,8124,8124,8124,8124,8124,8124,8124,8124,...,8124,8124,8124,8124,8124,8124,8124,8124,8124,8124
unique,6,4,10,2,9,2,2,2,12,2,...,9,9,1,4,3,5,9,6,7,2
top,x,y,n,f,n,f,c,b,b,t,...,w,w,p,w,o,p,w,v,d,e
freq,3656,3244,2284,4748,3528,7914,6812,5612,1728,4608,...,4464,4384,8124,7924,7488,3968,2388,4040,3148,4208


A continuación traspongo la tabla previa para que me muestre la tabla completa

In [ ]:
df.describe(include='all').T

,count,unique,top,freq
cap-shape,8124,6,x,3656
cap-surface,8124,4,y,3244
cap-color,8124,10,n,2284
bruises,8124,2,f,4748
odor,8124,9,n,3528
gill-attachment,8124,2,f,7914
gill-spacing,8124,2,c,6812
gill-size,8124,2,b,5612
gill-color,8124,12,b,1728
stalk-shape,8124,2,t,4608


#Vemos nuevamente que el stalk-root está incompleto. Necesito revisar qué significa para decidir si eliminarlo o no.

#Además, aquí encuentro una variable "veil-type", la cual solo tiene 1 unique value. Es decir que toda la columna es lo mismo y por lo mismo puede eliminarse.

De igual forma hay otras columnas donde una sola variable componen la mayoría de las filas, sin embargo estas sí pueden aportar información para determinar entre una clase y otra.

In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   cap-shape                 8124 non-null   object
 1   cap-surface               8124 non-null   object
 2   cap-color                 8124 non-null   object
 3   bruises                   8124 non-null   object
 4   odor                      8124 non-null   object
 5   gill-attachment           8124 non-null   object
 6   gill-spacing              8124 non-null   object
 7   gill-size                 8124 non-null   object
 8   gill-color                8124 non-null   object
 9   stalk-shape               8124 non-null   object
 10  stalk-root                5644 non-null   object
 11  stalk-surface-above-ring  8124 non-null   object
 12  stalk-surface-below-ring  8124 non-null   object
 13  stalk-color-above-ring    8124 non-null   object
 14  stalk-color-below-ring  

#Todos los datatypes se cambiaron a object (es bueno que toda la columna tenga el mismo tipo). En este dataset, no hay datos numéricos, solo clasificativos.

#Limpieza de datos

Eliminar datos basura y rellenar valores necesarios.

In [ ]:
conteo_no_nulos = df.count().sort_values(ascending=False)
total_filas = len(df)

resumen_completitud = pd.DataFrame({
    "No_nulos": conteo_no_nulos,
    "Total_filas": total_filas,
    "Porcentaje_completitud (%)": round(conteo_no_nulos / total_filas * 100, 2)
})

resumen_completitud = resumen_completitud.sort_values(by="Porcentaje_completitud (%)", ascending=False)

print("Resumen de completitud de columnas:")
display(resumen_completitud)

umbral = 70
columnas_bajas = resumen_completitud[resumen_completitud["Porcentaje_completitud (%)"] < umbral]
if not columnas_bajas.empty:
    print(f"\n Columnas con menos del {umbral}% de completitud:")
    display(columnas_bajas)
else:
    print(f"\nTodas las columnas tienen al menos {umbral}% de completitud.")


Resumen de completitud de columnas:


,No_nulos,Total_filas,Porcentaje_completitud (%)
cap-shape,8124,8124,100.00
cap-surface,8124,8124,100.00
cap-color,8124,8124,100.00
bruises,8124,8124,100.00
odor,8124,8124,100.00
gill-attachment,8124,8124,100.00
gill-spacing,8124,8124,100.00
gill-size,8124,8124,100.00
gill-color,8124,8124,100.00
stalk-shape,8124,8124,100.00



 Columnas con menos del 70% de completitud:


,No_nulos,Total_filas,Porcentaje_completitud (%)
stalk-root,5644,8124,69.47


#Como ya sabíamos, stalk root está incompleto. Sin embargo está en casi un 70% de los registros por lo que probablemente decida manejar dos sets de datos. Uno sin la columna y otro sin las filas incompletas para ver qué resultado es mejor. Por ahora, solo generaré el que elimina la columna por completo

##limpieza de columnas

In [ ]:
df.shape

(8124, 23)

In [ ]:
# Lista de columnas a eliminar
columnas_a_eliminar = [
    "stalk-root",
    "veil-type"
]

# Eliminar las columnas (si existen)
df_limpio = df.drop(columns=[col for col in columnas_a_eliminar if col in df.columns])

print("Columnas eliminadas correctamente.")

df_limpio.shape


Columnas eliminadas correctamente.


(8124, 21)

#Eliminé la columna incompleta únicamente en df_limpio

##Identificar valores únicos

El siguiente código es medio redundante porque describe() ya me dice los valores únicos de las columnas, pero me deja las columnas organizadas de menor variabilidad a mayor así que igual lo voy a dejar.

Cuanto menos me ayuda a ver que 6 de las columnas pueden ser binarias

In [ ]:
valores_unicos = df_limpio.nunique().sort_values()

resumen_unicos = pd.DataFrame({
    "Valores_únicos": valores_unicos,
    "Total_filas": len(df_limpio),
    "Porcentaje_únicos (%)": round(valores_unicos / len(df_limpio) * 100, 3)
})

# Mostrar de menor a mayor (columnas menos variables primero)
print("Cantidad de valores únicos por columna:")
display(resumen_unicos)

# === (Opcional) Mostrar solo columnas con muy poca variabilidad ===
umbral_unicos = 2  # por ejemplo, columnas con solo 1 o 2 valores únicos
columnas_poco_utiles = resumen_unicos[resumen_unicos["Valores_únicos"] <= umbral_unicos]

if not columnas_poco_utiles.empty:
    print(f"\n !!!!!!!!!!!!! Columnas con {umbral_unicos} o menos valores únicos:")
    display(columnas_poco_utiles)
else:
    print(f"\n Todas las columnas tienen más de {umbral_unicos} valores únicos.")


Cantidad de valores únicos por columna:


,Valores_únicos,Total_filas,Porcentaje_únicos (%)
bruises,2,8124,0.025
gill-size,2,8124,0.025
gill-spacing,2,8124,0.025
gill-attachment,2,8124,0.025
stalk-shape,2,8124,0.025
poisonous,2,8124,0.025
ring-number,3,8124,0.037
cap-surface,4,8124,0.049
veil-color,4,8124,0.049
stalk-surface-above-ring,4,8124,0.049



 !!!!!!!!!!!!! Columnas con 2 o menos valores únicos:


,Valores_únicos,Total_filas,Porcentaje_únicos (%)
bruises,2,8124,0.025
gill-size,2,8124,0.025
gill-spacing,2,8124,0.025
gill-attachment,2,8124,0.025
stalk-shape,2,8124,0.025
poisonous,2,8124,0.025


# En el siguiente código con dropna, se eliminan las filas con valores vacios. En este caso solo serían las de stalk root, pero eliminé la columna

In [ ]:
filas_antes = len(df_limpio)

df_limpio = df_limpio.dropna()

filas_despues = len(df_limpio)
filas_eliminadas = filas_antes - filas_despues

print(f" Se eliminaron {filas_eliminadas} filas con datos vacíos.")
print(f" Total antes: {filas_antes} → Total después: {filas_despues}")

df_limpio.to_excel("/content/datosHongosLimpios.xlsx", index=False)
print("Archivo final guardado como 'datosHongosLimpios.xlsx'")

 Se eliminaron 0 filas con datos vacíos.
 Total antes: 8124 → Total después: 8124
Archivo final guardado como 'datosHongosLimpios.xlsx'


In [ ]:
df_limpio.shape

(8124, 21)

#Separación de dataset en train y test

In [ ]:
from sklearn.model_selection import train_test_split
#Ahora sí separamos los datos
X = df_limpio.drop("poisonous", axis=1)
y = df_limpio["poisonous"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,

    #permite mantener proporciones de y (poisonous) en la separación
    stratify=y
)

#Tratamiento de datos

Código para agregar algunas columnas que pueden servir para el análisis y cambio en los formatos de datos.

Para este problema seguramente sea necesario hacer un one hot encoding

##Escalamiento (one hot encoding)

In [ ]:
X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)


print(X_train_encoded.shape)
print(X_test_encoded.shape)

(6499, 111)
(1625, 110)


Ambas columnas deberían ser iguales, por ello hay que alinearlas, ya que si no no se puede predecir.

Lo que hacemos es si en test hay (azul/amarillo) y en train hay (azul/rojo), vamos a tener en ambos (azul/amarillo/rojo)

In [ ]:
# Hacer que las columnas coincidan entre train y test
X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded,
    join='left',
    axis=1,
    fill_value=0
)

print(X_train_encoded.shape)
print(X_test_encoded.shape)

(6499, 111)
(1625, 111)


In [ ]:
X_train_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6499 entries, 5249 to 2411
Columns: 111 entries, cap-shape_b to habitat_w
dtypes: bool(111)
memory usage: 755.3 KB


In [ ]:
X_train_encoded.describe(include='all').T

,count,unique,top,freq
cap-shape_b,6499,2,False,6138
cap-shape_c,6499,2,False,6495
cap-shape_f,6499,2,False,3982
cap-shape_k,6499,2,False,5846
cap-shape_s,6499,2,False,6472
...,...,...,...,...
habitat_l,6499,2,False,5839
habitat_m,6499,2,False,6268
habitat_p,6499,2,False,5608
habitat_u,6499,2,False,6198
